# 04 - Stage 1 - corruption-contrast pretraining + zero-shot check

Pretrains the edit encoder on training-fold edits only (evaluation authors excluded, proof in disjointness.json), then scores real labelled changesets with the frozen pretext model. Report the zero-shot number whatever it is.

| | |
|---|---|
| Expected time | 20-60 min |
| Needs first | nothing |
| Runs | `python train/run_pipeline.py --feature-store data/interim --task <TASK> --stages 04` |

**Before running:** upload `mapguard_bundle.zip` (built with `python cluster/bundle.py`) next to
this notebook. Submit with the `tensorflow_pbs` image, 1 GPU, 8 CPUs, 32 GB. See RUN_ALTAIR.md.

Outputs land in `mapguard/experiments/runs/<stage>/...`; cell 4 zips them for download.
Every run directory carries its config, git SHA, data fingerprint and split file.

In [ ]:
# 1. Find the bundle, unpack it once, install only what is missing.
import glob, json, os, pathlib, subprocess, sys, zipfile

HOME = pathlib.Path.cwd()
if (HOME / "train" / "run_pipeline.py").exists():   # cell re-run after chdir below
    HOME = HOME.parent
candidates = sorted(glob.glob("mapguard_bundle*.zip") + glob.glob("../mapguard_bundle*.zip")
                    + glob.glob("/data/**/mapguard_bundle*.zip", recursive=True),
                    key=os.path.getmtime)
ROOT = HOME / "mapguard"
# Unpack when there is no copy yet, or when a newer bundle was uploaded. experiments/runs is
# never inside a bundle, so earlier results survive an update.
if candidates:
    with zipfile.ZipFile(candidates[-1]) as z:
        fresh = json.loads(z.read("BUNDLE_MANIFEST.json"))
        current = ROOT / "BUNDLE_MANIFEST.json"
        if not current.exists() or json.loads(current.read_text()).get("generated") != fresh.get("generated"):
            z.extractall(ROOT)
            print("unpacked", candidates[-1], "->", ROOT)
elif not (ROOT / "train" / "run_pipeline.py").exists():
    raise SystemExit("upload cluster/_bundles/mapguard_bundle.zip next to this notebook first")
os.chdir(ROOT)
manifest = json.loads((ROOT / "BUNDLE_MANIFEST.json").read_text())
print(json.dumps({k: manifest.get(k) for k in ("git_sha", "git_dirty", "generated", "feature_schema")}, indent=2))

# Install missing packages without letting pip upgrade numpy: the image's TensorFlow is
# built against its own numpy, and a silent upgrade breaks it.
import importlib.util, numpy
missing = [pkg for mod, pkg in (("pyarrow", "pyarrow"), ("pandas", "pandas"),
                                ("sklearn", "scikit-learn"), ("xgboost", "xgboost"))
           if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing,
                           f"numpy=={numpy.__version__}"])
import tensorflow as tf
print("python", sys.version.split()[0], "| tensorflow", tf.__version__,
      "| GPUs", [g.name for g in tf.config.list_physical_devices("GPU")])

In [ ]:
# 2. Run. QUICK=True is a smoke test with tiny epochs - never report its numbers.
TASK = "ovid"          # ovid | T1 | T2  (T1/T2 need corpora M and N in the feature store)
QUICK = False
STAGES = "04"

cmd = [sys.executable, "-u", "train/run_pipeline.py", "--feature-store", "data/interim",
       "--task", TASK, "--stages", STAGES] + (["--quick"] if QUICK else [])
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
print("exit code", proc.wait())

In [ ]:
# 3. Headlines of the runs this job produced.
summaries = sorted(glob.glob("experiments/pipeline/*.json"), key=os.path.getmtime)
summary = json.loads(pathlib.Path(summaries[-1]).read_text())
print(json.dumps(summary["stages"], indent=1))
for run_dir in summary["runs_created"]:
    res = pathlib.Path(run_dir) / "results.json"
    if res.exists():
        r = json.loads(res.read_text())
        print(f"\n{run_dir}\n  headline: {json.dumps(r.get('headline'))[:600]}")

In [ ]:
# 4. Pack results for download (Files -> right-click -> Download). Weights included.
import shutil, time
out = HOME / f"mapguard_results_{STAGES.replace(',', '-')}_{TASK}_{time.strftime('%Y%m%d-%H%M%S')}"
shutil.make_archive(str(out), "zip", root_dir=ROOT, base_dir="experiments")
print("download:", str(out) + ".zip")